# 07 — Mini Project: RAG With Your Own Files (Save, Reload & Benchmark)

**By the end of this notebook, you can index an arbitrary folder of your own documents, save & reload the vector index, and benchmark answers across 14 evaluation questions.**

### Learning Objectives
- Build a reusable `build_or_load_index(folder, index_path)` utility for personal files.
- Verify round-trip save/restart/reload queries return exact cited sources cleanly.
- Run the Phase 1 pipeline against `eval_questions.json` and achieve `>= 10/14` sensible answers/refusals.

### Prerequisites
- Notebooks 00–06

## 1. Setup
Load `.env` configuration via `course/utils/models.py` and initialize our shared LLM and embeddings helpers.

In [1]:
# If running in Google Colab, uncomment: !git clone https://github.com/rahat4528/rag-from-scratch-course-v2.git && %cd rag-from-scratch-course-v2 && pip install -q -r course/requirements.txt
import sys
from pathlib import Path
repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'course' / 'utils').exists()), Path.cwd())
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
from course.utils.models import get_llm, get_embeddings, resolve_course_root
course_dir = resolve_course_root(); data_dir = course_dir / 'data'
llm = get_llm(fallback_to_demo=True); embeddings = get_embeddings()
print(f'Ready | LLM={type(llm).__name__} | Embeddings={type(embeddings.base).__name__}')

[OfflineDemoChatModel] Notice: Provider 'ollama' not active (Ollama server is not reachable at http://localhost:11434. Install/start Ollama (`ollama pull llama3.2`) or switch RAG_PROVIDER in .env.). Running code-path demo mode.


W1112 00:38:23.225000 12752 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready | LLM=OfflineDemoChatModel | Embeddings=LocalSentenceTransformerEmbeddings


## 2. Concept: From Fixed Demo Files to Reusable Personal Workspace

Up to now we loaded Novasphere's sample folder. In real life, learners drop new `.txt`, `.md`, or `.pdf` reports into their own folder, **build the vector index once**, save it cleanly, and reload it in future sessions!

**Troubleshooting Checklist Before Trusting Answers:**
1. *No documents loaded?* Confirm UTF-8 encoding & non-empty files.
2. *Wrong answer retrieved?* Print top-k chunks BEFORE blaming the prompt; widen `chunk_size` (500–650) or adjust `k`.
3. *Model hallucinating?* Verify `<context>` delimiter + refusal wording in prompt.

```
Your Folder (.txt/.md/.pdf) --> [Load & Section Split] --> [Embed & Save JSON Index]
                                                                    |
(Later Restart Session) ---> [Load JSON Index Instantaneous] ---> Query + Citations!
```

## 3. Step-by-Step Build

### Step 1: Create a Custom User Folder & Persist Index to JSON
Let's simulate adding custom project notes into a temporary workspace folder alongside our corpus, then save the index to disk.

In [2]:
import json, tempfile
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from course.utils.helpers import load_folder_documents, print_qa_result

tmp_dir = tempfile.TemporaryDirectory()
user_folder = Path(tmp_dir.name) / "my_notes"; user_folder.mkdir()
(user_folder / "lab_notes.md").write_text("Project Apollo backup key is APOLLO-772 and uses port 9090.", encoding="utf-8")
index_file = str(Path(tmp_dir.name) / "saved_index.json")

user_docs = load_folder_documents(user_folder)
splitter = RecursiveCharacterTextSplitter(chunk_size=620, chunk_overlap=130, separators=["\n## ", "\n### ", "\n\n", "\n- ", "\n", " ", ""])
InMemoryVectorStore.from_documents(splitter.split_documents(user_docs), embeddings).dump(index_file)
print("Indexed custom files & persisted cleanly:", Path(index_file).name)
        

Indexed custom files & persisted cleanly: saved_index.json


> **What just happened?**
> Dropping any `.txt`, `.md`, or `.pdf` into `user_folder` indexes cleanly with one call.

### Step 2: Reload the Persisted Index & Ask a Cited Question
Now simulate a fresh script launch loading directly from `index_file` without re-splitting or re-embedding source files.

In [3]:
reloaded_user_store = InMemoryVectorStore.load(index_file, embeddings)
hits = reloaded_user_store.similarity_search("What port does Project Apollo use?", k=1)
ans = llm.invoke(f"<context>\n{hits[0].page_content}\n</context>\nQuestion: What port does Project Apollo use?").content
print_qa_result("What port does Project Apollo use?", ans, hits)
assert "9090" in ans and hits[0].metadata["source"] == "lab_notes.md"
        

Question : What port does Project Apollo use?
Answer   : Based on the provided context: Project Apollo backup key is APOLLO-772 and uses port 9090.
Sources  : lab_notes.md


> **What just happened?**
> Persisted reload works cleanly with exact citation `lab_notes.md`.

### Step 3: Index Full Course Corpus & Evaluate Across All 14 Benchmark Questions
Now let's test our Phase 1 core pipeline (`chunk_size=620`, section separators, `k=4`) across all 14 `eval_questions.json` items (`>= 10/14` target).

In [4]:
from course.utils.helpers import load_corpus_documents
corpus_chunks = splitter.split_documents(load_corpus_documents(data_dir))
corpus_store = InMemoryVectorStore.from_documents(corpus_chunks, embeddings)
eval_qs = json.loads((data_dir / "eval_questions.json").read_text(encoding="utf-8"))

passed = 0
for q in eval_qs:
    top_d = corpus_store.similarity_search(q["question"], k=4)
    ctx = "\n\n".join(f"[{d.metadata['source']}]\n{d.page_content}" for d in top_d)
    a = llm.invoke(f"<context>\n{ctx}\n</context>\nQuestion: {q['question']}").content.lower().replace("$", "")
    ok = ("don't know" in a or "not" in a) if q["type"] == "unanswerable" else all(kf.lower().replace("$", "") in a for kf in q["key_facts"])
    passed += int(ok)
    print(f"[{'PASS' if ok else 'MISS'}] {q['id']}: {q['question'][:48]}")
print(f"\nPhase 1 Core Score: {passed}/{len(eval_qs)}")
assert passed >= 10, f"Expected >= 10/14, got {passed}"
        

[PASS] q01: How much does the Enterprise Plan cost per month
[PASS] q02: By what percentage did HyperSync Auto-Tuning red
[PASS] q03: What is the maximum daily meal expense reimburse
[PASS] q04: What uptime SLA percentage is guaranteed for the
[PASS] q05: Which subscription tier is required to use ZeroG
[PASS] q06: If a customer on the Professional tier experienc
[PASS] q07: Who was appointed Chief AI Ethics Officer in Oct
[PASS] q08: On which days are pets allowed in the Zurich off
[PASS] q09: Which ROS2 distributions are natively supported 
[PASS] q10: What steps should be taken to resolve the error 
[PASS] q11: When will the legacy REST API v1 be shut down co
[PASS] q12: What is Novasphere Dynamics' stock ticker symbol
[PASS] q13: What is the warranty and return policy for physi
[PASS] q14: What is the monthly pricing for Novasphere's sat

Phase 1 Core Score: 14/14


> **What just happened?**
> Even with Phase 1 dense retrieval alone, we cleared the `>= 10/14` Phase 1 milestone — and we will inspect any misses carefully in Phase 2 & 3!

## 4. Try It Yourself
1. **Clean up temp dir safely** — *Hint:* Call `tmp_dir.cleanup()` cleanly.
2. **Inspect source hit rate** — *Hint:* Check how many answerable questions had the ground-truth file in `k=4`.

<details>
<summary><strong>Click to reveal sample solution</strong></summary>
Run the cell below to check one working solution.
</details>

In [5]:
tmp_dir.cleanup()
ans_qs = [q for q in eval_qs if q["type"] != "unanswerable"]
src_hits = sum(1 for q in ans_qs if any(s in d.metadata["source"] for d in corpus_store.similarity_search(q["question"], k=4) for s in (q.get("source_docs") or [q["source_doc"]])))
print(f"Answerable source Hit@4 = {src_hits}/{len(ans_qs)}")
        

Answerable source Hit@4 = 11/11


## 5. Common Mistakes
- Forgetting to re-run index dump after editing files inside your personal folder.
- Saving vector stores without keeping matching embedding config in `.env`.
- Accepting low scores without debugging which exact question type missed (numbers vs multi-doc vs unanswerable).

## 6. Recap
- A clean folder loader + `.dump()` / `.load()` lets beginners chat with any custom folder.
- Systematic troubleshooting starts by inspecting retrieved chunks before tweaking prompts.
- Evaluating against 14 structured questions gives an objective Phase 1 baseline before adding Phase 2–3 upgrades.

## 7. Next Notebook
Next (`08_query_translation.ipynb`, Phase 2!): Fix ambiguous & compound queries using Multi-Query, RAG-Fusion (RRF), Decomposition, Step-Back, and HyDE!

## 8. Video Chapter Outline
1. **Indexing Arbitrary Personal Folders** — Supporting custom `.txt`/`.md`/`.pdf` sets.
2. **Save, Restart & Reload Pattern** — Persistent JSON vector store.
3. **3-Step Troubleshooting Checklist** — Encoding -> Chunks -> Prompt.
4. **14-Question Baseline Evaluation** — Measuring our Phase 1 milestone objectively.
5. **Phase 1 Complete!** — Previewing Advanced Query Translation.